# otro_pipe/02 — features causales nuevas + hojas de Random Forest + poda shadow

Segunda vuelta sobre `01_Pipeline_completo.ipynb`: la grilla de 20 combos
corrida contra datos reales mostro sobreajuste (brecha val->test de hasta
+0.25 en varios combos). Este notebook ataca eso con: dos features nuevas
CAUSALES (tiempo que tardo el cliente en adoptar el producto, tasa de
recompra), productos magicos como FEATURE (no como filtro, eso ya lo hace
`01_`), hojas de un Random Forest como features categoricas (tree
embedding), y una poda de features por "shadow features" (columnas de
ruido puro: cualquier feature real con menos importancia que la mejor
columna de ruido se descarta) antes de correr la MISMA grilla de 20 combos
de `01_`.

Reusa el preprocesamiento y la cache de `01_` tal cual (no se recalcula el
cartesiano), y reusa integramente el motor de Optuna/grilla/submit — lo
nuevo esta todo en el feature engineering, antes de la grilla.


## 0) Setup


In [ ]:
import gc, json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import lightgbm as lgb
import optuna
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


def _leer_json_reintentando(path, intentos=5, espera=2):
    """El bucket es un mount GCS FUSE: a veces tira Input/output error transitorio."""
    ultimo_error = None
    for _ in range(intentos):
        try:
            with open(path, encoding="utf-8") as f:
                return json.load(f)
        except OSError as e:
            ultimo_error = e
            time.sleep(espera)
    raise ultimo_error


BUCKET    = resolver_bucket()
DIR_RAW   = BUCKET / "datasets"
DIR_PRE   = BUCKET / "datasets_pre"      # cache del cartesiano -- la ESCRIBE 01_
DIR_FE    = BUCKET / "datasets_fe"       # cache de FE + productos_magicos.json
RUTA_EXP  = BUCKET / "exp_otro_pipe"     # MISMA carpeta que 01_ -- comparten leaderboard
for d in (DIR_PRE, DIR_FE, RUTA_EXP):
    d.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"crudos : {DIR_RAW}")
print(f"cache pre: {DIR_PRE}")
print(f"cache FE : {DIR_FE}")
print(f"salida   : {RUTA_EXP}")


def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


def a_indice_mes(p: int) -> int:
    return (p // 100) * 12 + (p % 100) - 1


def desplazar_meses(p: int, k: int) -> int:
    m = a_indice_mes(p) + k
    return (m // 12) * 100 + (m % 12) + 1


In [ ]:
PARAM = {
    # ── Deben coincidir con 01_ (para leer su cache del cartesiano) ───────
    'solo_productos_target': False,
    'horizonte': 2,
    'max_lags': 12,
    'ventanas_ma': (3, 6, 12),
    'ventanas_racha': (3, 6),
    'niveles_share': ('cat1', 'cat2', 'cat3', 'mercado'),
    'lags_share': 3,
    'techo_indice': 10.0,
    # Con el esquema 'febreros' (ver abajo), MESES_TRAIN termina justo antes
    # del febrero mas reciente disponible -- 201905 quedaba despues de ese
    # corte y rompia el chequeo de causalidad. 201712 es conservador.
    'mes_corte_vecinos': 201712,
    'n_vecinos': 3,

    # ── Particion train/val/test: esquema 'febreros' (igual que 01_) ──────
    # El objetivo real (periodo_objetivo) es SIEMPRE febrero. En vez de un
    # bloque de meses cualquiera, VAL es el febrero historico mas reciente
    # disponible (mismo mes calendario que el objetivo real) y TRAIN es todo
    # lo anterior. Se calcula en la seccion de split, no aca -- ver
    # periodos_con_target_en_mes().
    'reentrenar_con_val_para_test': True,

    # ── Filtro del universo de entrenamiento: SOLO volumen (80% demanda).
    # Productos magicos esta vez es feature, no filtro -- ver seccion 4.
    'pct_demanda_cubrir': 0.80,
    # 'productos_magicos.json' = calculado por residuos_2 (WAPE en val).
    # 'productos_magicos_profesor.json' = ayuda de la catedra (no calculado,
    # guardarlo aparte -- ver residuos_2.ipynb, seccion de productos magicos).
    'archivo_productos_magicos': 'productos_magicos_profesor.json',

    # ── Hojas de Random Forest (NUEVO) ─────────────────────────────────
    'n_arboles_rf': 50,
    'profundidad_rf': 6,
    'min_hoja_rf': 50,

    # ── Poda por shadow features (NUEVO) ───────────────────────────────
    'n_shadow': 10,

    # ── La grilla (identica a 01_) ─────────────────────────────────────
    'escalados': ('mediana', 'media', 'zscore', 'rolling_mean', 'rango', 'normalpower'),
    'ventana_escalado_rolling': 3,
    'targets_escalado_nativos': ('ton_norm', 'delta_ton_norm', 'log_ton_norm'),
    'targets_fijos': ('delta_mean_12', 'delta_reg'),
    'salto_delta': None,     # None -> usa 'horizonte'
    'ridge_alpha': 1.0,

    # ── Optuna por combo ────────────────────────────────────────────────
    'n_trials': 30,
    'backup_cada_n_trials': 10,
    'regularizacion': 'normal',
    'techo_arboles': 500,
    'early_stopping_rounds': 50,

    # ── Entrega ─────────────────────────────────────────────────────────
    'periodo_objetivo': 202002,
    'semilla_ensemble': [102191],
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': False,
    'pausa_entre_submits_seg': 2,

    # Prefijo de experimento -- distinto al de 01_ para no pisar sus
    # resultado.json (ambos quedan en la MISMA exp_otro_pipe/).
    'prefijo_experimento': 'otropipeV2',

    'semilla': 102191,
    'sufijo': '',
}
if PARAM['salto_delta'] is None:
    PARAM['salto_delta'] = PARAM['horizonte']

H = PARAM['horizonte']
L = PARAM['max_lags']
KEYS = ['product_id', 'customer_id']
KEYS_SQL = ", ".join(KEYS)

GRILLA = ([(t, e) for t in PARAM['targets_escalado_nativos'] for e in PARAM['escalados']]
         + [(t, None) for t in PARAM['targets_fijos']])
print(f"grilla: {len(GRILLA)} combos (identica a 01_)")


## 1) Preprocesamiento — se lee la cache que ya escribio `01_`

No se recalcula: si `01_` no corrio todavia, hay que correrlo primero (al
menos hasta que escriba `datasets_pre/sellin_zeroes_...otroPipe.parquet`).


In [ ]:
_tgt = '_tgtFilter' if PARAM['solo_productos_target'] else ''
NOMBRE_PRE = f"sellin_zeroes_grpClienteProducto{_tgt}_otroPipe.parquet"
path_pre = DIR_PRE / NOMBRE_PRE

if not path_pre.exists():
    raise FileNotFoundError(
        f"No encontre {path_pre}. Corre 01_Pipeline_completo.ipynb primero "
        f"(al menos hasta que termine el preprocesamiento) -- este notebook "
        f"reusa esa cache, no la recalcula."
    )
print(f"preprocesamiento reusado de 01_: {path_pre.name}")


## 2) FE extendido — igual a `01_` + 2 features causales nuevas

Mismo cuerpo de `02_FE` de `01_Pipeline_completo.ipynb` (shares, lags,
medias moviles, deltas, indices, racha, recencia, peso acumulado, vecinos),
mas:

- **`tiempo_adquisicion_causal`**: meses entre el nacimiento del PRODUCTO
  (`m_nace_prod`) y la primera compra REAL de ese cliente. NULL en las
  filas anteriores a esa primera compra (todavia no paso, no se puede
  saber), fijo desde ahi en adelante -- causal por construccion.
- **`win_rate_causal`**: meses con compra de ESE producto (acumulado,
  causal) / meses que el cliente estuvo activo comprando cualquier cosa
  (acumulado, causal, desde que lo conoce).

Cache en un parquet NUEVO (`..._v2features.parquet`), no pisa la cache de
`01_`.


In [ ]:
NOMBRE_SIN_ESCALAR_V2 = (f"features_sin_escalar_grpClienteProducto{_tgt}_{L}lags_share_{H}h"
                        f"_vec{PARAM['n_vecinos']}_otroPipeV2features.parquet")
path_fe = DIR_FE / NOMBRE_SIN_ESCALAR_V2

if path_fe.exists():
    print(f"cache encontrada: {path_fe.name} -> se salta el FE")
else:
    t0 = time.time()
    con = duckdb.connect()
    con.execute(f"""
        CREATE OR REPLACE TABLE panel_raw AS
        SELECT * FROM read_parquet('{path_pre}')
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE prod AS
        SELECT * FROM read_csv('{DIR_RAW / "tb_productos.txt"}', delim='\t', header=true)
    """)
    n_raw = con.sql("SELECT COUNT(*) FROM panel_raw").fetchone()[0]
    print(f"preprocesado: {n_raw:,} filas   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE panel_0 AS
        SELECT {KEYS_SQL}, periodo,
               SUM(tn) AS tn,
               SUM(cust_request_tn) AS req_tn,
               SUM(cust_request_qty) AS req_qty,
               MAX(plan_precios_cuidados) AS precios_cuidados,
               ((periodo // 100) * 12 + (periodo % 100)) AS m
        FROM panel_raw GROUP BY {KEYS_SQL}, periodo
    """)
    n0 = con.sql("SELECT COUNT(*) FROM panel_0").fetchone()[0]
    rango = con.sql("SELECT MIN(periodo), MAX(periodo) FROM panel_0").fetchone()
    print(f"panel agregado: {n0:,} filas · rango {rango[0]} -> {rango[1]}   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE panel_1 AS
        SELECT p.*,
               MIN(m) OVER (PARTITION BY {KEYS_SQL}) AS m_nace,
               MAX(m) OVER (PARTITION BY {KEYS_SQL}) AS m_muere,
               pr.cat1, pr.cat2, pr.cat3, pr.brand, pr.sku_size
        FROM panel_0 p LEFT JOIN prod pr USING (product_id)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE panel_1 AS
        SELECT *, CASE WHEN m >= m_nace THEN m - m_nace ELSE -1 END AS edad_cliente_producto
        FROM panel_1
    """)
    n_ceros = con.sql("SELECT COUNT(*) FROM panel_1 WHERE tn = 0").fetchone()[0]
    print(f"ceros de tn: {n_ceros:,} ({100*n_ceros/n0:.0f}%)   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE tot_prod AS
        SELECT product_id, m, SUM(tn) AS tn_prod, COUNT(*) AS n_clientes_prod
        FROM panel_1 GROUP BY product_id, m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE tot_cli AS
        SELECT customer_id, m, SUM(tn) AS tn_cli, COUNT(*) AS n_productos_cli
        FROM panel_1 GROUP BY customer_id, m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE vida_prod AS
        SELECT product_id, MIN(m) AS m_nace_prod FROM tot_prod GROUP BY product_id
    """)
    con.execute("""
        CREATE OR REPLACE TABLE panel_1 AS
        SELECT p.*,
               CASE WHEN p.m >= v.m_nace_prod THEN p.m - v.m_nace_prod ELSE -1 END AS edad_producto
        FROM panel_1 p LEFT JOIN vida_prod v USING (product_id)
    """)

    for niv in PARAM['niveles_share']:
        if niv == 'mercado':
            con.execute("""
                CREATE OR REPLACE TABLE tot_mercado AS
                SELECT m, SUM(tn_prod) AS tn_mercado FROM tot_prod GROUP BY m
            """)
        else:
            con.execute(f"""
                CREATE OR REPLACE TABLE tot_{niv} AS
                SELECT pr.{niv} AS {niv}, tp.m, SUM(tp.tn_prod) AS tn_{niv}, COUNT(*) AS n_prod_{niv}
                FROM tot_prod tp LEFT JOIN prod pr USING (product_id)
                GROUP BY pr.{niv}, tp.m
            """)
    print(f"tot_prod: {con.sql('SELECT COUNT(*) FROM tot_prod').fetchone()[0]:,} filas   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT p.*, tp.tn_prod, tp.n_clientes_prod, tc.tn_cli, tc.n_productos_cli,
               CASE WHEN ABS(tc.tn_cli) > 1e-9 THEN p.tn / tc.tn_cli ELSE 0.0 END AS sh_prod_en_cli,
               CASE WHEN ABS(tp.tn_prod) > 1e-9 THEN p.tn / tp.tn_prod ELSE 0.0 END AS sh_cli_en_prod
        FROM panel_1 p
        LEFT JOIN tot_prod tp USING (product_id, m)
        LEFT JOIN tot_cli tc USING (customer_id, m)
    """)
    SHARES = ["sh_prod_en_cli", "sh_cli_en_prod"]

    for niv in PARAM['niveles_share']:
        if niv == 'mercado':
            con.execute("""
                CREATE OR REPLACE TABLE df AS
                SELECT d.*, tm.tn_mercado,
                       CASE WHEN ABS(tm.tn_mercado) > 1e-9 THEN d.tn_prod / tm.tn_mercado ELSE 0.0 END AS sh_prod_en_mercado
                FROM df d LEFT JOIN tot_mercado tm USING (m)
            """)
            SHARES.append("sh_prod_en_mercado")
        else:
            con.execute(f"""
                CREATE OR REPLACE TABLE df AS
                SELECT d.*, t.tn_{niv},
                       CASE WHEN ABS(t.tn_{niv}) > 1e-9 THEN d.tn_prod / t.tn_{niv} ELSE 0.0 END AS sh_prod_en_{niv}
                FROM df d LEFT JOIN tot_{niv} t ON d.{niv} = t.{niv} AND d.m = t.m
            """)
            SHARES.append(f"sh_prod_en_{niv}")
    print(f"{len(SHARES)} shares: {SHARES}   [{time.time()-t0:.0f}s]")

    mn, mx = con.sql("""
        SELECT MIN(s), MAX(s) FROM (
            SELECT customer_id, m, SUM(sh_prod_en_cli) AS s FROM df GROUP BY 1, 2
        )
    """).fetchone()
    assert abs(mx - 1.0) < 1e-6, "sh_prod_en_cli no suma 1 en algun cliente-mes"
    if 'cat3' in PARAM['niveles_share']:
        mn3, mx3 = con.sql("""
            SELECT MIN(s), MAX(s) FROM (
                SELECT cat3, m, SUM(sh_prod_en_cat3) AS s
                FROM (SELECT DISTINCT product_id, cat3, m, sh_prod_en_cat3 FROM df)
                GROUP BY 1, 2
            )
        """).fetchone()
        assert abs(mx3 - 1.0) < 1e-6, "sh_prod_en_cat3 no suma 1 en algun cat3-mes"
    print("chequeo de shares OK")

    t0 = time.time()
    lag_exprs = [f"LAG(tn, {k}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS tn_lag{k}"
                for k in range(1, L + 1)]
    ma_exprs = [f"AVG(tn) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
               f"ROWS BETWEEN {w-1} PRECEDING AND CURRENT ROW) AS tn_ma{w}"
               for w in PARAM['ventanas_ma']]
    qty_ma_exprs = [f"AVG(req_qty) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
                   f"ROWS BETWEEN {w-1} PRECEDING AND CURRENT ROW) AS qty_ma{w}"
                   for w in PARAM['ventanas_ma'][:2]]
    otros = [
        f"LAG(req_qty, 1) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS qty_lag1",
        f"AVG(req_tn) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
        f"ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS reqtn_ma3",
        f"MAX(tn) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
        f"ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS tn_pico_hasta_aca",
    ]
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(lag_exprs + ma_exprs + qty_ma_exprs + otros)} FROM df")
    print(f"lags 1..{L} + medias moviles {PARAM['ventanas_ma']} agregados   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    delta_exprs = []
    for s in SHARES:
        delta_exprs += [f"LAG({s}, {k}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS {s}_lag{k}"
                        for k in range(1, PARAM['lags_share'] + 1)]
        delta_exprs += [f"AVG({s}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m "
                        f"ROWS BETWEEN {w-1} PRECEDING AND CURRENT ROW) AS {s}_ma{w}"
                        for w in (3, 6)]
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(delta_exprs)} FROM df")

    d_exprs = []
    for s in SHARES:
        d_exprs.append(f"({s} - {s}_lag1) AS {s}_d1")
        d_exprs.append(f"({s} - {s}_ma3) AS {s}_dma3")
        if PARAM['lags_share'] >= 3:
            d_exprs.append(f"({s} - {s}_lag3) AS {s}_d3")
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(d_exprs)} FROM df")

    TECHO = PARAM['techo_indice']
    idx_exprs = [
        f"CASE WHEN ABS(tn_lag1) > 1e-9 THEN LEAST(GREATEST(tn / tn_lag1, 0.0), {TECHO}) ELSE NULL END AS idx_tn_mom",
        f"CASE WHEN ABS(tn_ma3) > 1e-9 THEN LEAST(GREATEST(tn / tn_ma3, 0.0), {TECHO}) ELSE NULL END AS idx_tn_vs_ma3",
        f"CASE WHEN ABS(tn_pico_hasta_aca) > 1e-9 THEN LEAST(GREATEST(tn / tn_pico_hasta_aca, 0.0), {TECHO}) ELSE NULL END AS idx_tn_vs_pico",
        f"CASE WHEN ABS(qty_lag1) > 1e-9 THEN LEAST(GREATEST(req_qty / qty_lag1, 0.0), {TECHO}) ELSE NULL END AS idx_qty_mom",
    ]
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(idx_exprs)} FROM df")
    print(f"deltas de share + 4 indices agregados. [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("CREATE OR REPLACE TABLE df AS SELECT *, CAST(tn > 0 AS TINYINT) AS vendio FROM df")
    con.execute(f"""
        CREATE OR REPLACE TABLE df AS
        SELECT *,
               AVG(CAST(vendio AS DOUBLE)) OVER (PARTITION BY {KEYS_SQL} ORDER BY m
                   ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS frac_meses_con_venta_6,
               SUM(vendio) OVER (PARTITION BY {KEYS_SQL} ORDER BY m
                   ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS meses_con_venta_3,
               (periodo % 100) AS mes_del_anio,
               CAST(edad_cliente_producto BETWEEN 0 AND 6 AS TINYINT) AS es_nuevo
        FROM df
    """)

    def _racha_expr(w):
        terminos = []
        for k in range(1, w + 1):
            factores = ["vendio"] + [
                f"COALESCE(LAG(vendio, {j}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m), 0)"
                for j in range(1, k)
            ]
            terminos.append("(" + " * ".join(factores) + ")")
        return " + ".join(terminos)

    racha_exprs = [f"({_racha_expr(w)}) AS racha_{w}" for w in PARAM['ventanas_racha']]
    con.execute(f"CREATE OR REPLACE TABLE df AS SELECT *, {', '.join(racha_exprs)} FROM df")
    print(f"racha consecutiva agregada: {[f'racha_{w}' for w in PARAM['ventanas_racha']]}   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE df AS
        SELECT *,
               LAST_VALUE(CASE WHEN tn > 0 THEN m END IGNORE NULLS) OVER (
                   PARTITION BY {KEYS_SQL} ORDER BY m
                   ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
               ) AS _ultimo_m_con_venta_prev
        FROM df
    """)
    con.execute("CREATE OR REPLACE TABLE df AS SELECT * EXCLUDE (_ultimo_m_con_venta_prev), "
               "(m - _ultimo_m_con_venta_prev) AS meses_sin_compra FROM df")
    print(f"meses_sin_compra agregado.   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE tn_total_mes AS
        SELECT m, SUM(tn) AS tn_total_mes,
               SUM(SUM(tn)) OVER (ORDER BY m ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS tn_total_acum
        FROM panel_1 GROUP BY m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE tot_prod_acum AS
        SELECT tp.product_id, tp.m,
               CASE WHEN ABS(tm.tn_total_acum) > 1e-9
                    THEN SUM(tp.tn_prod) OVER (PARTITION BY tp.product_id ORDER BY tp.m
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / tm.tn_total_acum
                    ELSE 0.0 END AS peso_producto_acum
        FROM tot_prod tp LEFT JOIN tn_total_mes tm USING (m)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*, tpa.peso_producto_acum
        FROM df d LEFT JOIN tot_prod_acum tpa USING (product_id, m)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE tot_cli_acum AS
        SELECT tc.customer_id, tc.m,
               CASE WHEN ABS(tm.tn_total_acum) > 1e-9
                    THEN SUM(tc.tn_cli) OVER (PARTITION BY tc.customer_id ORDER BY tc.m
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / tm.tn_total_acum
                    ELSE 0.0 END AS peso_cliente_acum
        FROM tot_cli tc LEFT JOIN tn_total_mes tm USING (m)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*, tca.peso_cliente_acum
        FROM df d LEFT JOIN tot_cli_acum tca USING (customer_id, m)
    """)
    print(f"peso_producto_acum y peso_cliente_acum agregados.   [{time.time()-t0:.0f}s]")

    # ── NUEVO: win_rate_causal (tasa de recompra) ──────────────────────────
    # meses_con_venta_acum (par, causal) / meses_activo_cliente_acum (cliente,
    # causal) -- ambos ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW, asi
    # que en la fila de un mes m solo cuentan meses <= m (nunca el futuro).
    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE df AS
        SELECT *,
               SUM(vendio) OVER (PARTITION BY {KEYS_SQL} ORDER BY m
                   ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS meses_con_venta_acum
        FROM df
    """)
    con.execute("""
        CREATE OR REPLACE TABLE cli_vendio_acum AS
        SELECT customer_id, m,
               SUM(CAST(tn_cli > 0 AS TINYINT)) OVER (PARTITION BY customer_id ORDER BY m
                   ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS meses_activo_cliente_acum
        FROM tot_cli
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*, COALESCE(cva.meses_activo_cliente_acum, 0) AS meses_activo_cliente_acum
        FROM df d LEFT JOIN cli_vendio_acum cva USING (customer_id, m)
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT *,
               CASE WHEN meses_activo_cliente_acum > 0
                    THEN CAST(meses_con_venta_acum AS DOUBLE) / meses_activo_cliente_acum
                    ELSE NULL END AS win_rate_causal
        FROM df
    """)
    print(f"win_rate_causal agregado (meses_con_venta_acum / meses_activo_cliente_acum).   [{time.time()-t0:.0f}s]")

    # ── NUEVO: tiempo_adquisicion_causal ───────────────────────────────────
    # primera compra REAL del par, vista SOLO hasta la fila actual (NULL si
    # todavia no compro nunca a esa altura) menos el nacimiento del producto.
    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*,
               MIN(CASE WHEN d.tn > 0 THEN d.m END) OVER (PARTITION BY {KEYS_SQL} ORDER BY d.m
                   ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS _primera_compra_causal
        FROM df d
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*,
               CASE WHEN d._primera_compra_causal IS NOT NULL
                    THEN d._primera_compra_causal - v.m_nace_prod ELSE NULL END AS tiempo_adquisicion_causal
        FROM df d LEFT JOIN vida_prod v USING (product_id)
    """)
    print(f"tiempo_adquisicion_causal agregado (NULL antes de la primera compra real).   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    M_CORTE = (PARAM['mes_corte_vecinos'] // 100) * 12 + (PARAM['mes_corte_vecinos'] % 100)
    N_VEC = PARAM['n_vecinos']

    wide = (con.sql(f"""
                SELECT product_id, m, tn_prod FROM tot_prod WHERE m < {M_CORTE}
            """).pl()
               .pivot(on="product_id", index="m", values="tn_prod")
               .sort("m")
               .drop("m"))
    corr = wide.to_pandas().corr(method="spearman")

    vecinos_rows = []
    for p in corr.columns:
        s = corr[p].drop(labels=[p]).dropna()
        if s.empty:
            continue
        for rank, (vecino, r) in enumerate(s.sort_values().head(N_VEC).items(), start=1):
            vecinos_rows.append({"product_id": p, "tipo": "sustituto", "rank": rank,
                                 "vecino_id": vecino, "corr": float(r)})
        for rank, (vecino, r) in enumerate(s.sort_values(ascending=False).head(N_VEC).items(), start=1):
            vecinos_rows.append({"product_id": p, "tipo": "complementario", "rank": rank,
                                 "vecino_id": vecino, "corr": float(r)})

    vecinos_df = pd.DataFrame(vecinos_rows)
    con.register("vecinos_pl", vecinos_df)
    con.execute("""
        CREATE OR REPLACE TABLE vecinos AS
        SELECT CAST(product_id AS BIGINT) AS product_id, tipo, rank,
               CAST(vecino_id AS BIGINT) AS vecino_id, corr
        FROM vecinos_pl
    """)
    con.unregister("vecinos_pl")
    print(f"vecinos calculados para {con.sql('SELECT COUNT(DISTINCT product_id) FROM vecinos').fetchone()[0]} "
         f"productos (corte {PARAM['mes_corte_vecinos']})   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute("""
        CREATE OR REPLACE TABLE feat_vecinos AS
        SELECT v.product_id, v.tipo, tp.m, AVG(tp.tn_prod) AS tn_vecino_prom
        FROM vecinos v LEFT JOIN tot_prod tp ON tp.product_id = v.vecino_id
        GROUP BY v.product_id, v.tipo, tp.m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE feat_vecinos_piv AS
        SELECT product_id, m,
               MAX(CASE WHEN tipo = 'sustituto' THEN tn_vecino_prom END) AS tn_sustitutos_prom,
               MAX(CASE WHEN tipo = 'complementario' THEN tn_vecino_prom END) AS tn_complementarios_prom
        FROM feat_vecinos GROUP BY product_id, m
    """)
    con.execute("""
        CREATE OR REPLACE TABLE df AS
        SELECT d.*,
               COALESCE(fv.tn_sustitutos_prom, 0.0) AS tn_sustitutos_prom,
               COALESCE(fv.tn_complementarios_prom, 0.0) AS tn_complementarios_prom
        FROM df d LEFT JOIN feat_vecinos_piv fv USING (product_id, m)
    """)
    print(f"tn_sustitutos_prom / tn_complementarios_prom agregados.   [{time.time()-t0:.0f}s]")

    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE df AS
        SELECT *,
               LEAD(tn, {H}) OVER (PARTITION BY {KEYS_SQL} ORDER BY m) AS clase_tn,
               (((m + {H} - 1) // 12) * 100) + ((m + {H} - 1) % 12) + 1 AS periodo_objetivo
        FROM df
    """)
    _sup = con.sql("SELECT COUNT(*) FROM df WHERE clase_tn IS NOT NULL").fetchone()[0]
    _tot = con.sql("SELECT COUNT(*) FROM df").fetchone()[0]
    print(f"filas con target: {_sup:,}   filas de inferencia: {_tot - _sup:,}   [{time.time()-t0:.0f}s]")

    errores = []
    def chk(ok, msg):
        print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
        if not ok:
            errores.append(msg)

    print("CONTROL DE DATA LEAKAGE (FE)")
    print("=" * 74)
    _una = con.sql(f"""
        SELECT {KEYS_SQL} FROM df WHERE clase_tn IS NOT NULL
        GROUP BY {KEYS_SQL} ORDER BY COUNT(*) DESC LIMIT 1
    """).fetchone()
    _k = dict(zip(KEYS, _una))
    _where = " AND ".join(f"{c} = {v}" for c, v in _k.items())
    _serie = con.sql(f"SELECT m, tn, clase_tn FROM df WHERE {_where} ORDER BY m").pl()
    _tn, _cl = _serie["tn"].to_list(), _serie["clase_tn"].to_list()
    _malos = [i for i in range(len(_tn) - H)
             if _cl[i] is not None and abs(_cl[i] - _tn[i + H]) > 1e-9]
    chk(not _malos, f"clase_tn[i] == tn[i+{H}] en la serie {_k}")

    info = con.sql("DESCRIBE df").fetchall()
    tipos = {r[0]: r[1].upper().split("(")[0] for r in info}
    PROHIBIDAS = {"clase_tn", "periodo_objetivo", "m_muere", "m", "periodo", "m_nace"} | set(KEYS)
    FEATURES_FE = [c for c in tipos if c not in PROHIBIDAS]
    chk(not (set(FEATURES_FE) & {"clase_tn", "periodo_objetivo"}), "el target no esta entre las features")
    chk("m_muere" not in FEATURES_FE, "m_muere no es feature")
    chk("m_nace" not in FEATURES_FE, "m_nace no es feature")

    NUMERIC = {"TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT", "UTINYINT",
              "USMALLINT", "UINTEGER", "UBIGINT", "FLOAT", "DOUBLE", "DECIMAL"}
    NUM_FEATURES = [c for c in FEATURES_FE if tipos[c] in NUMERIC]
    _sel = ", ".join(f'CORR(clase_tn, "{c}") AS "{c}"' for c in NUM_FEATURES)
    _res = con.sql(f"SELECT {_sel} FROM df WHERE clase_tn IS NOT NULL").fetchone()
    _sosp = [(c, round(v, 5)) for c, v in zip(NUM_FEATURES, _res) if v is not None and abs(v) > 0.999]
    chk(not _sosp, f"ninguna de las {len(NUM_FEATURES)} features numericas correlaciona >0.999 con clase_tn  {_sosp}")

    _s2 = con.sql(f"SELECT m, tn, tn_ma3 FROM df WHERE {_where} ORDER BY m").pl()
    _tn2, _ma = _s2["tn"].to_list(), _s2["tn_ma3"].to_list()
    _err_ma = max((abs(_ma[i] - sum(_tn2[i-2:i+1]) / 3)
                  for i in range(2, len(_tn2)) if _ma[i] is not None), default=0.0)
    chk(_err_ma < 1e-9, f"tn_ma3[t] == promedio(tn[t-2..t]): error maximo {_err_ma:.2e}")

    _s3 = con.sql(f"SELECT m, vendio, racha_3 FROM df WHERE {_where} ORDER BY m").pl()
    _v3, _r3 = _s3["vendio"].to_list(), _s3["racha_3"].to_list()
    _r3_manual = []
    _run = 0
    for v in _v3:
        _run = (_run + 1) if v else 0
        _r3_manual.append(min(_run, 3))
    _err_racha = max((abs(a - b) for a, b in zip(_r3, _r3_manual)), default=0)
    chk(_err_racha == 0, f"racha_3 coincide con el conteo manual de racha consecutiva (error max {_err_racha})")

    # ── NUEVO: chequeo de causalidad de tiempo_adquisicion_causal ─────────
    _s4 = con.sql(f"SELECT m, tn, tiempo_adquisicion_causal FROM df WHERE {_where} ORDER BY m").pl()
    _m4, _tn4, _tad4 = _s4["m"].to_list(), _s4["tn"].to_list(), _s4["tiempo_adquisicion_causal"].to_list()
    _primera_real = next((m for m, tn in zip(_m4, _tn4) if tn > 0), None)
    if _primera_real is not None:
        _antes_ok = all(v is None for m, v in zip(_m4, _tad4) if m < _primera_real)
        _despues_fijo = len({v for m, v in zip(_m4, _tad4) if m >= _primera_real}) <= 1
        chk(_antes_ok, "tiempo_adquisicion_causal es NULL en todas las filas ANTES de la primera compra real")
        chk(_despues_fijo, "tiempo_adquisicion_causal queda FIJO desde la primera compra real en adelante")
    else:
        print("  [info ] la serie de ejemplo nunca compro -- se salta el chequeo de tiempo_adquisicion_causal")

    # ── NUEVO: chequeo de causalidad de meses_con_venta_acum / win_rate_causal ──
    _s5 = con.sql(f"SELECT m, vendio, meses_con_venta_acum FROM df WHERE {_where} ORDER BY m").pl()
    _v5, _mca5 = _s5["vendio"].to_list(), _s5["meses_con_venta_acum"].to_list()
    _acum_manual, _run5 = [], 0
    for v in _v5:
        _run5 += v
        _acum_manual.append(_run5)
    _err_acum = max((abs(a - b) for a, b in zip(_mca5, _acum_manual)), default=0)
    chk(_err_acum == 0, f"meses_con_venta_acum coincide con el conteo manual causal (error max {_err_acum})")

    _s6 = con.sql(f"""
        SELECT meses_con_venta_acum, meses_activo_cliente_acum, win_rate_causal FROM df
        WHERE {_where} AND meses_activo_cliente_acum > 0 ORDER BY m
    """).pl()
    if _s6.height:
        _wr_manual = (_s6["meses_con_venta_acum"].cast(pl.Float64)
                     / _s6["meses_activo_cliente_acum"].cast(pl.Float64)).to_list()
        _wr_real = _s6["win_rate_causal"].cast(pl.Float64).to_list()
        _err_wr = max((abs(a - b) for a, b in zip(_wr_manual, _wr_real)), default=0)
        chk(_err_wr < 1e-9, f"win_rate_causal = meses_con_venta_acum / meses_activo_cliente_acum (error max {_err_wr:.2e})")

    # (el chequeo de que mes_corte_vecinos no pise MESES_TRAIN se hace en la
    # seccion de split -- con el esquema 'febreros' el fin de train no se
    # conoce hasta ahi, no aca)

    print("=" * 74)
    if errores:
        raise RuntimeError(f"Control de leakage FALLIDO: {errores}")
    print(f"Control superado. {len(FEATURES_FE)} features ({len(NUM_FEATURES)} numericas).")

    t0 = time.time()
    DROP = {"m_nace", "m_muere", "_primera_compra_causal"}
    CTX_F64 = {"clase_tn", "tn0"}
    exprs = []
    for name, tipo, *_r in info:
        if name in DROP:
            continue
        tipo_base = tipo.upper().split("(")[0]
        out_name = "tn0" if name == "tn" else name
        if tipo_base == "DOUBLE" and out_name not in CTX_F64:
            exprs.append(f'CAST("{name}" AS FLOAT) AS "{out_name}"')
        else:
            exprs.append(f'"{name}" AS "{out_name}"')
    select_sql = ",\n       ".join(exprs)

    con.execute(f"""
        COPY (SELECT {select_sql} FROM df ORDER BY {KEYS_SQL}, m)
        TO '{path_fe}' (FORMAT parquet)
    """)
    con.close()
    print(f"Guardado: {path_fe}   {_tot:,} filas x {len(exprs)} columnas   [{time.time()-t0:.0f}s]")


## 3) Carga + split train/val/test/infer + control de leakage

Identico a `01_` (portado tal cual).


In [ ]:
t0 = time.time()
CTX_F64 = {'clase_tn', 'tn0'}

lf = pl.scan_parquet(path_fe)
_schema = lf.collect_schema()
COLS_ALL = list(_schema.keys())
_f64 = [c for c, t in _schema.items() if t == pl.Float64]
_a_f32 = [c for c in _f64 if c not in CTX_F64]
lf = lf.with_columns([pl.col(c).cast(pl.Float32) for c in _a_f32])

periodos = sorted(lf.select('periodo').unique().collect()['periodo'].to_list())
print(f"Dataset: {len(COLS_ALL)} columnas   Periodos: {periodos[0]} -> {periodos[-1]} ({len(periodos)} meses)")

MESES_INFER = periodos[-H:]
df_infer = lf.filter(pl.col('clase_tn').is_null() & pl.col('periodo').is_in(MESES_INFER)).collect()
df_sup = lf.filter(pl.col('clase_tn').is_not_null()).collect()
print(f"Supervisadas: {df_sup.height:,}   Inferencia: {df_infer.height:,} -> "
     f"periodos {sorted(df_infer['periodo'].unique().to_list())}")
print(f"[{time.time()-t0:.0f}s]")

periodos_sup = sorted(df_sup['periodo'].unique().to_list())


def periodos_con_target_en_mes(periodos_disponibles, horizonte, mes_objetivo):
    """Periodos p (de periodos_disponibles) tales que p+horizonte cae en el
    mes calendario 'mes_objetivo' (1-12) Y ese p+horizonte tambien tiene datos
    reales (para poder medir WAPE contra un target de verdad)."""
    disp = set(periodos_disponibles)
    out = [p for p in periodos_disponibles
          if desplazar_meses(p, horizonte) in disp
          and desplazar_meses(p, horizonte) % 100 == mes_objetivo]
    return sorted(out)


MES_OBJETIVO = PARAM['periodo_objetivo'] % 100
_candidatos_feb = periodos_con_target_en_mes(periodos_sup, H, MES_OBJETIVO)
print(f"esquema de validacion: 'febreros' (mes objetivo {MES_OBJETIVO}, "
     f"de periodo_objetivo={PARAM['periodo_objetivo']})")
print(f"periodos con target en mes {MES_OBJETIVO} disponibles: {_candidatos_feb}")
if not _candidatos_feb:
    raise RuntimeError(
        f"No hay ningun periodo con target en el mes {MES_OBJETIVO}. "
        f"Revisa el rango de datos o PARAM['horizonte']."
    )

MESES_VAL   = [_candidatos_feb[-1]]
_train_max = desplazar_meses(MESES_VAL[0], -H)
MESES_TRAIN = [p for p in periodos_sup if p <= _train_max]

_test_candidatos = [p for p in periodos_sup
                    if a_indice_mes(p) - a_indice_mes(MESES_VAL[0]) >= H]
if not _test_candidatos:
    raise RuntimeError(f"No hay periodos disponibles para TEST con gap >= horizonte "
                       f"desde VAL ({MESES_VAL[0]}).")
MESES_TEST = [_test_candidatos[-1]]

for nombre, ms in (('TRAIN', MESES_TRAIN), ('VAL', MESES_VAL), ('TEST', MESES_TEST)):
    if not ms:
        raise ValueError(f"{nombre} quedo vacio. Disponibles: {periodos_sup[0]}..{periodos_sup[-1]}")
    print(f"{nombre:6s} ({len(ms):2d} meses): {ms[0]} .. {ms[-1]}")

if PARAM['mes_corte_vecinos'] > max(MESES_TRAIN):
    raise RuntimeError(f"mes_corte_vecinos ({PARAM['mes_corte_vecinos']}) supera el fin de "
                       f"MESES_TRAIN ({max(MESES_TRAIN)}) -- los vecinos por correlacion "
                       f"estarian mirando datos de val/test.")

leak = {'errores': [], 'ok': []}
def _err(msg):
    leak['errores'].append(msg); print(f"  [ERROR] {msg}")
def _ok(msg):
    leak['ok'].append(msg); print(f"  [ok]    {msg}")

print("CONTROL DE DATA LEAKAGE (split)")
print("=" * 72)
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, 'train', 'val'), (MESES_VAL, MESES_TEST, 'val', 'test')):
    gap = a_indice_mes(min(b)) - a_indice_mes(max(a))
    (_ok if gap >= H else _err)(f"gap {na}({max(a)}) -> {nb}({min(b)}) = {gap} mes(es)"
                                + ("" if gap >= H else f" < horizonte {H}"))
if PARAM['reentrenar_con_val_para_test']:
    gap_tv = a_indice_mes(min(MESES_TEST)) - a_indice_mes(max(MESES_TRAIN + MESES_VAL))
    (_ok if gap_tv >= H else _err)(f"gap (train+val) -> test = {gap_tv}" + ("" if gap_tv >= H else f" < {H}"))
for (na, a), (nb, b) in ((('train', MESES_TRAIN), ('val', MESES_VAL)),
                        (('train', MESES_TRAIN), ('test', MESES_TEST)),
                        (('val', MESES_VAL),     ('test', MESES_TEST))):
    inter = sorted(set(a) & set(b))
    (_err if inter else _ok)(f"{na} y {nb}" + (f" comparten {inter}" if inter else " son disjuntos"))
if max(MESES_TRAIN) < min(MESES_VAL) and max(MESES_VAL) < min(MESES_TEST):
    _ok("orden cronologico correcto: train < val < test")
else:
    _err("orden cronologico incorrecto")
periodos_infer = set(df_infer['periodo'].unique().to_list())
solapa = sorted(periodos_infer & (set(MESES_TRAIN) | set(MESES_VAL) | set(MESES_TEST)))
(_err if solapa else _ok)("periodos de inferencia" + (f" solapan: {solapa}" if solapa else " no se usan para entrenar/validar/testear"))
print("=" * 72)
if leak['errores']:
    raise RuntimeError(f"Control de leakage FALLIDO: {leak['errores']}")
print(f"Control superado. {df_sup.height:,} filas supervisadas, {df_infer.height:,} de inferencia.")


## 4) Universo de entrenamiento (SOLO 80% de demanda) + `es_magico` como feature

A diferencia de `01_`, aca los productos magicos NO filtran filas -- se
agregan como una columna booleana mas y el modelo decide que hacer con esa
senal. El filtro que SI se mantiene es el de volumen (80% de demanda
acumulada), igual que en `01_`.


In [ ]:
t0 = time.time()

demanda_pares = (df_sup.group_by(['customer_id', 'product_id'])
                       .agg(pl.col('tn0').sum().alias('tn_total'))
                       .sort('tn_total', descending=True)
                       .with_columns(
                           (pl.col('tn_total').cum_sum() - pl.col('tn_total')).alias('_acum_antes')))
_total_demanda = demanda_pares['tn_total'].sum()
demanda_pares = demanda_pares.with_columns(
    (pl.col('_acum_antes') / _total_demanda if _total_demanda > 0 else pl.lit(0.0)).alias('_frac_acum_antes'))
pares_universo = demanda_pares.filter(pl.col('_frac_acum_antes') < PARAM['pct_demanda_cubrir'])

print(f"pares totales: {demanda_pares.height:,}   "
     f"pares que cubren el {100*PARAM['pct_demanda_cubrir']:.0f}% de la demanda: {pares_universo.height:,} "
     f"({100*pares_universo.height/demanda_pares.height:.0f}%)")
print(f"demanda cubierta por esos pares: "
     f"{100*pares_universo['tn_total'].sum()/_total_demanda:.1f}%   [{time.time()-t0:.0f}s]")

if pares_universo.height == 0:
    raise RuntimeError("El universo de entrenamiento (80% de demanda) quedo vacio. "
                       "Revisa PARAM['pct_demanda_cubrir'].")

UNIVERSO = pares_universo.select(['customer_id', 'product_id']).unique()
df_sup_universo = df_sup.join(UNIVERSO, on=['customer_id', 'product_id'], how='inner')
df_infer_universo = df_infer.join(UNIVERSO, on=['customer_id', 'product_id'], how='inner')
print(f"\ndf_sup_universo: {df_sup_universo.height:,} filas (de {df_sup.height:,})")
print(f"df_infer_universo: {df_infer_universo.height:,} filas (de {df_infer.height:,})")

FALLBACK_INFER = (df_infer.join(UNIVERSO, on=['customer_id', 'product_id'], how='anti')
                          .select('product_id', 'customer_id', 'periodo', 'periodo_objetivo',
                                  pl.col('tn_ma3').fill_null(0.0).alias('tn_pred_fallback')))
print(f"fallback (fuera del universo, inferencia): {FALLBACK_INFER.height:,} filas")

# ── es_magico como FEATURE (no como filtro -- eso ya lo hace 01_) ────────
PRODUCTOS_MAGICOS = None
if PARAM['archivo_productos_magicos']:
    path_mag = DIR_FE / PARAM['archivo_productos_magicos']
    if not path_mag.exists():
        print(f"no encontre {path_mag} -- corre nat_exp/residuos_2.ipynb primero. "
             f"es_magico queda en False para todos.")
    else:
        _meta_mag = _leer_json_reintentando(path_mag)
        PRODUCTOS_MAGICOS = set(_meta_mag['product_ids'])
        print(f"productos magicos leidos de {path_mag.name}: {len(PRODUCTOS_MAGICOS)}")

_es_magico_expr = (pl.col('product_id').is_in(PRODUCTOS_MAGICOS or set())
                  .cast(pl.Utf8).alias('es_magico'))
# Utf8 en vez de Boolean: polars no puede castear Boolean -> Categorical
# directo (lo necesitamos mas adelante, mismo patron que el resto de las
# categoricas), y como texto se auto-detecta como categorica sin problema.
df_sup_universo = df_sup_universo.with_columns(_es_magico_expr)
df_infer_universo = df_infer_universo.with_columns(_es_magico_expr)
_n_magicas = int((df_sup_universo['es_magico'] == 'true').sum())
print(f"es_magico agregado como feature ({_n_magicas:,} de "
     f"{df_sup_universo.height:,} filas de train/val/test son de productos magicos)   [{time.time()-t0:.0f}s]")


## 5) Hojas de Random Forest (NUEVO) — tree embedding

Un Random Forest chico (arboles poco profundos, hoja minima alta para no
explotar la cardinalidad categorica) entrenado UNICAMENTE en `MESES_TRAIN`
(causal: val/test/infer nunca participan del fiteo). El indice de hoja
donde cae cada fila, por arbol, se agrega como columna categorica nueva —
tecnica clasica tipo GBDT+LR (Facebook, 2014).


In [ ]:
t0 = time.time()
from sklearn.ensemble import RandomForestRegressor

COLS_ID = [c for c in ['product_id', 'customer_id', 'periodo', 'm', 'periodo_objetivo']
          if c in df_sup_universo.columns]
PROHIBIDAS_BASE = set(COLS_ID) | {'clase_tn'}
FEATURES_BASE = [c for c in df_sup_universo.columns if c not in PROHIBIDAS_BASE]
CAT_PEDIDAS = ['cat1', 'cat2', 'cat3', 'brand']
TIPOS_TEXTO = (pl.Utf8, pl.String, pl.Categorical, pl.Enum, pl.Boolean)
CAT_BASE = ([c for c in CAT_PEDIDAS if c in FEATURES_BASE]
           + [c for c in FEATURES_BASE if df_sup_universo.schema[c] in TIPOS_TEXTO and c not in CAT_PEDIDAS])
print(f"FEATURES base (FE + causales nuevas + es_magico, antes de hojas/shadow): {len(FEATURES_BASE)}")

PARAM_RF = dict(n_estimators=PARAM['n_arboles_rf'], max_depth=PARAM['profundidad_rf'],
               min_samples_leaf=PARAM['min_hoja_rf'], n_jobs=-1, random_state=PARAM['semilla'])

_num_rf = [c for c in FEATURES_BASE if c not in CAT_BASE]
_tr_rf = df_sup_universo.filter(pl.col('periodo').is_in(MESES_TRAIN))
X_rf_train = _tr_rf.select(_num_rf).fill_null(0.0).to_numpy()
y_rf_train = _tr_rf['clase_tn'].fill_null(0.0).to_numpy()

rf = RandomForestRegressor(**PARAM_RF)
rf.fit(X_rf_train, y_rf_train)
print(f"Random Forest entrenado: {PARAM_RF['n_estimators']} arboles x {len(_num_rf)} features "
     f"numericas, {X_rf_train.shape[0]:,} filas de TRAIN.   [{time.time()-t0:.0f}s]")


def agregar_hojas(df_pl):
    X = df_pl.select(_num_rf).fill_null(0.0).to_numpy()
    hojas = rf.apply(X)
    return df_pl.with_columns([pl.Series(f"hoja_arbol_{i}", hojas[:, i].astype(str))
                               for i in range(hojas.shape[1])])


df_sup_universo = agregar_hojas(df_sup_universo)
df_infer_universo = agregar_hojas(df_infer_universo)
COLS_HOJA = [f"hoja_arbol_{i}" for i in range(PARAM_RF['n_estimators'])]
print(f"{len(COLS_HOJA)} columnas de hoja agregadas (categoricas, una por arbol).   [{time.time()-t0:.0f}s]")


## 6) Poda por shadow features (NUEVO) — DOS rondas, repetidas, mayoria de votos

Dos problemas reales encontrados (y corregidos) armando esta seccion,
documentados aca porque son la parte mas delicada del notebook:

1. Meter las features base Y las hojas de RF a competir en la MISMA
   carrera de importancia descartaba casi todo, incluido `tn0` -- las hojas
   son CASI un proxy directo de `clase_tn` (el RF se entreno prediciendola)
   y le "roban" el gain a cualquier feature cruda. Arreglado con DOS
   competencias separadas (features base vs sus propias shadow; hojas vs
   las suyas).
2. Con un solo LightGBM de 200 arboles/63 hojas sobre pocas filas, el
   propio modelo de diagnostico sobreajusta: encuentra splits espurios
   sobre las columnas de ruido y les da importancia alta -- ni siquiera
   `tn0` le ganaba a la mejor shadow en una sola corrida. Arreglado con un
   modelo mas chico/regularizado (menos hojas, `min_child_samples`
   proporcional al tamaño del set) y **5 repeticiones con shadow distintas
   cada vez**, exigiendo que una feature le gane al ruido en la MAYORIA de
   las repeticiones (no solo una vez) -- mismo criterio que el algoritmo
   Boruta original, mucho mas robusto a que una sola corrida salga
   ruidosa.


In [ ]:
t0 = time.time()
rng = np.random.default_rng(PARAM['semilla'])
N_SHADOW = PARAM['n_shadow']
N_REPETICIONES = 5
UMBRAL_MAYORIA = 0.5   # sobrevive si gana al ruido en > 50% de las repeticiones

_tr_base = df_sup_universo.filter(pl.col('periodo').is_in(MESES_TRAIN))
_va_base = df_sup_universo.filter(pl.col('periodo').is_in(MESES_VAL))
_n_tr_shadow = _tr_base.height


def _cols_ruido(n, prefijo):
    return [pl.Series(f"{prefijo}_{i}",
                      rng.normal(size=n) if i % 2 == 0 else rng.uniform(-1, 1, size=n))
           for i in range(N_SHADOW)]


def _sobreviven_por_shadow(features_candidatas, cats_candidatas, prefijo_shadow):
    """N_REPETICIONES corridas de LightGBM (chico y regularizado, para no
    sobreajustar en datasets chicos) con features_candidatas + shadow nuevas
    en cada repeticion. Devuelve la lista de features que le ganan a la
    mejor shadow en mas de la mitad de las repeticiones."""
    conteos = {c: 0 for c in features_candidatas}
    _min_hoja = max(20, _n_tr_shadow // 40)
    for rep in range(N_REPETICIONES):
        cols_shadow = [f"{prefijo_shadow}{rep}_{i}" for i in range(N_SHADOW)]
        tr = _tr_base.with_columns(_cols_ruido(_tr_base.height, f"{prefijo_shadow}{rep}"))
        va = _va_base.with_columns(_cols_ruido(_va_base.height, f"{prefijo_shadow}{rep}"))

        pool = features_candidatas + cols_shadow
        cats_pool = [c for c in cats_candidatas if c in pool]

        def _a_pandas_pool(df_pl):
            out = (df_pl.select(pool + ['clase_tn'])
                        .with_columns([pl.col(c).cast(pl.Categorical) for c in cats_pool])
                        .to_pandas())
            for c in cats_pool:
                out[c] = out[c].astype('category')
            return out

        df_pd_tr, df_pd_va = _a_pandas_pool(tr), _a_pandas_pool(va)
        for c in cats_pool:
            df_pd_va[c] = df_pd_va[c].cat.set_categories(df_pd_tr[c].cat.categories)

        modelo = lgb.LGBMRegressor(objective='regression', metric='mae', verbosity=-1,
                                   n_estimators=100, learning_rate=0.05, num_leaves=15,
                                   min_child_samples=_min_hoja, reg_alpha=1.0, reg_lambda=1.0,
                                   seed=PARAM['semilla'] + rep, n_jobs=-1)
        modelo.fit(df_pd_tr[pool], df_pd_tr['clase_tn'], categorical_feature=cats_pool,
                  eval_set=[(df_pd_va[pool], df_pd_va['clase_tn'])],
                  callbacks=[lgb.early_stopping(20, verbose=False)])

        imp = pd.Series(modelo.booster_.feature_importance(importance_type='gain'), index=pool)
        piso = float(imp[cols_shadow].max())
        for c in features_candidatas:
            if imp.get(c, 0.0) > piso:
                conteos[c] += 1

    sobreviven = [c for c in features_candidatas if conteos[c] / N_REPETICIONES > UMBRAL_MAYORIA]
    return sobreviven, conteos


# ── Ronda 1: features base (sin hojas) ────────────────────────────────
FEATURES_SOBREVIVEN_BASE, _conteos_base = _sobreviven_por_shadow(FEATURES_BASE, CAT_BASE, '_shadowA')
_descartadas_base = [c for c in FEATURES_BASE if c not in FEATURES_SOBREVIVEN_BASE]
print(f"[ronda 1: features base] {N_REPETICIONES} repeticiones -- "
     f"candidatas: {len(FEATURES_BASE)}   sobreviven (mayoria): {len(FEATURES_SOBREVIVEN_BASE)}")
print(f"  tn0 gano al ruido en {_conteos_base.get('tn0', 0)}/{N_REPETICIONES} repeticiones"
     + ("" if 'tn0' in FEATURES_SOBREVIVEN_BASE else "  <- AVISO: no sobrevivio, revisar"))

# ── Ronda 2: hojas de RF (sin las features base compitiendo) ──────────
FEATURES_SOBREVIVEN_HOJA, _conteos_hoja = _sobreviven_por_shadow(COLS_HOJA, COLS_HOJA, '_shadowB')
print(f"[ronda 2: hojas de RF] {N_REPETICIONES} repeticiones -- "
     f"candidatas: {len(COLS_HOJA)}   sobreviven (mayoria): {len(FEATURES_SOBREVIVEN_HOJA)}")

FEATURES = FEATURES_SOBREVIVEN_BASE + FEATURES_SOBREVIVEN_HOJA
CAT_FEATURES = [c for c in (CAT_BASE + COLS_HOJA) if c in FEATURES]

if not FEATURES:
    raise RuntimeError("Ninguna feature sobrevivio la poda por shadow -- algo esta mal "
                       "(revisar N_REPETICIONES/UMBRAL_MAYORIA o el tamaño del dataset).")

print(f"\nFEATURES final: {len(FEATURES)} ({len(FEATURES_SOBREVIVEN_BASE)} base + "
     f"{len(FEATURES_SOBREVIVEN_HOJA)} hojas)")
if _descartadas_base:
    print(f"descartadas de la base: {_descartadas_base[:25]}"
         + (" ..." if len(_descartadas_base) > 25 else ""))
print(f"[{time.time()-t0:.0f}s]")


## 7) Los 5 targets (identico a `01_`)

Mismo esquema afin `norm = (valor-B0)/B1` -- portado tal cual, no depende
de que features sobrevivieron la poda.


In [ ]:
LAG_RAW = ["tn0"] + [f"tn_lag{k}" for k in range(1, L + 1)]

B1_EPSILON = 1e-3


def _calcular_b0_b1(metodo, lag_cols, ma_col):
    if metodo == "mediana":
        return pl.lit(0.0), pl.concat_list(lag_cols).list.median()
    if metodo == "media":
        return pl.lit(0.0), pl.mean_horizontal(lag_cols)
    if metodo == "zscore":
        return pl.mean_horizontal(lag_cols), pl.concat_list(lag_cols).list.std()
    if metodo == "rolling_mean":
        return pl.lit(0.0), pl.col(ma_col)
    if metodo == "rango":
        _l = pl.concat_list(lag_cols)
        return _l.list.min(), _l.list.max() - _l.list.min()
    if metodo == "normalpower":
        _l = pl.concat_list(lag_cols)
        p25 = _l.list.eval(pl.element().quantile(0.25)).list.first()
        p75 = _l.list.eval(pl.element().quantile(0.75)).list.first()
        return _l.list.median(), p75 - p25
    raise ValueError(f"metodo de escalado no soportado: {metodo}")


def preparar_target_escalado(df_in, target_name, metodo):
    usa_log = target_name == 'log_ton_norm'
    out = df_in
    if usa_log:
        out = out.with_columns([pl.col(c).clip(lower_bound=0.0).log1p().alias(f"__log_{c}")
                                for c in LAG_RAW])
        out = out.with_columns(pl.col(f"tn_ma{PARAM['ventana_escalado_rolling']}")
                               .clip(lower_bound=0.0).log1p().alias('__log_tn_ma'))
        lag_cols, ma_col = [f"__log_{c}" for c in LAG_RAW], '__log_tn_ma'
        clase_src, tn0_src = pl.col('clase_tn').clip(lower_bound=0.0).log1p(), pl.col('__log_tn0')
    else:
        lag_cols, ma_col = LAG_RAW, f"tn_ma{PARAM['ventana_escalado_rolling']}"
        clase_src, tn0_src = pl.col('clase_tn'), pl.col('tn0')

    B0, B1 = _calcular_b0_b1(metodo, lag_cols, ma_col)
    out = out.with_columns(B0.alias('B0'), B1.alias('B1'))
    b1_safe = (pl.when((pl.col('B1').abs() < B1_EPSILON) | pl.col('B1').is_null())
                 .then(1.0).otherwise(pl.col('B1')))
    out = out.with_columns(
        ((tn0_src - pl.col('B0')) / b1_safe).alias('_tn0_norm'),
        ((clase_src - pl.col('B0')) / b1_safe).alias('_clase_norm'))

    if target_name == 'delta_ton_norm':
        out = out.with_columns((pl.col('_clase_norm') - pl.col('_tn0_norm')).alias('y'))
        kind = 'delta_log' if usa_log else 'delta_nivel'
    else:
        out = out.with_columns(pl.col('_clase_norm').alias('y'))
        kind = 'log' if usa_log else 'nivel'
    return out, kind


def preparar_target_delta_mean12(df_in):
    out = df_in.with_columns(
        pl.col('tn_ma12').fill_null(0.0).alias('B0'),
        pl.lit(1.0).alias('B1'),
    ).with_columns((pl.col('clase_tn') - pl.col('B0')).alias('y'))
    return out, 'residuo_fijo'


COLS_LIN_REG = ["tn0"] + [f"tn_lag{k}" for k in range(1, L + 1)] + ["tn_ma3", "tn_ma6"]


def fit_ridge_delta_reg(df_train_pl):
    from sklearn.linear_model import Ridge
    X = df_train_pl.select(COLS_LIN_REG).fill_null(0.0).to_numpy()
    y = df_train_pl['clase_tn'].to_numpy()
    r = Ridge(alpha=PARAM['ridge_alpha'])
    r.fit(X, y)
    return r


def preparar_target_delta_reg(df_in, ridge_model):
    X = df_in.select(COLS_LIN_REG).fill_null(0.0).to_numpy()
    b0 = np.maximum(ridge_model.predict(X), 0.0)
    out = df_in.with_columns(pl.Series('B0', b0), pl.lit(1.0).alias('B1'))
    out = out.with_columns((pl.col('clase_tn') - pl.col('B0')).alias('y'))
    return out, 'residuo_fijo'


def reconstruir_target(pred, df_ctx, kind) -> np.ndarray:
    pred = np.asarray(pred, dtype=np.float64)
    if kind == 'residuo_fijo':
        return np.maximum(pred + df_ctx['B0'].to_numpy().astype(np.float64), 0.0)
    if kind.startswith('delta_'):
        pred = pred + df_ctx['_tn0_norm'].to_numpy().astype(np.float64)
        kind = kind[len('delta_'):]
    B0 = df_ctx['B0'].to_numpy().astype(np.float64)
    B1 = df_ctx['B1'].to_numpy().astype(np.float64)
    B1_safe = np.where((np.abs(B1) < B1_EPSILON) | ~np.isfinite(B1), 1.0, B1)
    val = pred * B1_safe + B0
    if kind == 'log':
        val = np.expm1(np.clip(val, -20.0, 20.0))
    return np.maximum(val, 0.0)


print("funciones de target listas")


### Chequeo round-trip de los 20 combos


In [ ]:
t0 = time.time()
_muestra = df_sup_universo.filter(pl.col('periodo').is_in(MESES_TRAIN)).head(2000)
_ridge_chk = fit_ridge_delta_reg(_muestra)

print(f"{'target':16s} {'escalado':14s} {'error max reconstruccion':>26s}")
print("-" * 60)
for target_name, metodo in GRILLA:
    if target_name in PARAM['targets_escalado_nativos']:
        out, kind = preparar_target_escalado(_muestra, target_name, metodo)
    elif target_name == 'delta_mean_12':
        out, kind = preparar_target_delta_mean12(_muestra)
    elif target_name == 'delta_reg':
        out, kind = preparar_target_delta_reg(_muestra, _ridge_chk)
    else:
        raise ValueError(target_name)
    rec = reconstruir_target(out['y'].to_numpy(), out, kind)
    err = float(np.max(np.abs(rec - out['clase_tn'].clip(lower_bound=0.0).to_numpy())))
    print(f"{target_name:16s} {str(metodo):14s} {err:26.2e}")
    assert err < 1e-6, f"round-trip roto para target={target_name} escalado={metodo} (kind={kind}): error {err}"
print(f"\n20/20 combos reconstruyen a toneladas con error <1e-6.   [{time.time()-t0:.0f}s]")


## 8) Motor de Optuna + WAPE (identico a `01_`)

`FEATURES`/`CAT_FEATURES` YA estan definidos (seccion 6, post-poda) -- esta
seccion solo define las funciones de entrenamiento/evaluacion, no las
recalcula.


In [ ]:
print(f"FEATURES (post-poda): {len(FEATURES)}   CAT_FEATURES: {len(CAT_FEATURES)}")


def wape(y_real_tn, y_pred_tn, product_ids=None, por_producto=True) -> float:
    y_real = np.asarray(y_real_tn, dtype=np.float64)
    y_pred = np.maximum(np.asarray(y_pred_tn, dtype=np.float64), 0.0)
    if por_producto and product_ids is not None:
        ids = np.asarray(product_ids)
        orden, inv = np.unique(ids, return_inverse=True)
        y_real = np.bincount(inv, weights=y_real, minlength=len(orden))
        y_pred = np.bincount(inv, weights=y_pred, minlength=len(orden))
    den = np.abs(y_real).sum()
    return float('nan') if den == 0 else float(np.abs(y_real - y_pred).sum() / den)


def espacio_hiper(trial, techo_arboles, regularizacion, semilla):
    base = {
        'objective': 'regression', 'metric': 'mae', 'verbosity': -1,
        'boosting_type': 'gbdt', 'seed': semilla, 'subsample_freq': 1, 'n_jobs': -1,
        'n_estimators': int(techo_arboles),
    }
    if regularizacion == 'fuerte':
        base.update({
            'num_leaves': trial.suggest_int('num_leaves', 8, 64),
            'max_depth': trial.suggest_int('max_depth', 3, 7),
            'learning_rate': trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 30, 200),
            'subsample': trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        })
    else:
        base.update({
            'num_leaves': trial.suggest_int('num_leaves', 20, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        })
    return base


def entrenar(df_pd_sub, params, meses_tr, eval_meses=None, early_stopping_rounds=50):
    idx = df_pd_sub.index[df_pd_sub['periodo'].isin(meses_tr)]
    if len(idx) == 0:
        raise ValueError(f'Sin filas de entrenamiento para los meses {meses_tr}')
    X, y = df_pd_sub.loc[idx, FEATURES], df_pd_sub.loc[idx, 'y']
    modelo = lgb.LGBMRegressor(**params)
    fit_kwargs = dict(categorical_feature=CAT_FEATURES)
    if eval_meses:
        idx_ev = df_pd_sub.index[df_pd_sub['periodo'].isin(eval_meses)]
        if len(idx_ev) == 0:
            raise ValueError(f'Sin filas de evaluacion para early stopping en {eval_meses}')
        fit_kwargs['eval_set'] = [(df_pd_sub.loc[idx_ev, FEATURES], df_pd_sub.loc[idx_ev, 'y'])]
        fit_kwargs['callbacks'] = [lgb.early_stopping(early_stopping_rounds, verbose=False)]
    modelo.fit(X, y, **fit_kwargs)
    del X, y
    gc.collect()
    return modelo


def evaluar_combo(modelo, df_pd_sub, meses_ev, kind):
    df_ev = df_pd_sub[df_pd_sub['periodo'].isin(meses_ev)]
    if len(df_ev) == 0:
        return float('nan'), None
    pred_y = modelo.predict(df_ev[FEATURES])
    pred_tn = reconstruir_target(pred_y, df_ev, kind)
    score = wape(df_ev['clase_tn'], pred_tn, df_ev['product_id'])
    return score, pred_tn


print("motor de entrenamiento listo")


## 9) `correr_combo()` — identico a `01_`, con prefijo de experimento distinto

Resume a nivel combo igual que `01_`: si `resultado.json` ya existe, se
saltea entero. Los nombres llevan el prefijo `otropipeV2` para no pisar los
resultados de `01_` (ambos quedan en la misma `exp_otro_pipe/`).


In [ ]:
oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")

_cols_pd = sorted(set(FEATURES + COLS_ID + ['clase_tn', 'B0', 'B1', '_tn0_norm']))
_cats = [c for c in CAT_FEATURES if c in _cols_pd]


def _a_pandas(df_pl, cols):
    cols_ok = [c for c in cols if c in df_pl.columns]
    out = (df_pl.select(cols_ok)
                .with_columns([pl.col(c).cast(pl.Categorical) for c in _cats if c in cols_ok])
                .to_pandas())
    for c in _cats:
        if c in out.columns:
            out[c] = out[c].astype('category')
    return out


def preparar_combo(target_name, metodo, df_train_base, df_infer_base):
    if target_name in PARAM['targets_escalado_nativos']:
        tr_out, kind = preparar_target_escalado(df_train_base, target_name, metodo)
        inf_out, _ = preparar_target_escalado(df_infer_base, target_name, metodo)
    elif target_name == 'delta_mean_12':
        tr_out, kind = preparar_target_delta_mean12(df_train_base)
        inf_out, _ = preparar_target_delta_mean12(df_infer_base)
    elif target_name == 'delta_reg':
        _ridge = fit_ridge_delta_reg(df_train_base.filter(pl.col('periodo').is_in(MESES_TRAIN)))
        tr_out, kind = preparar_target_delta_reg(df_train_base, _ridge)
        inf_out, _ = preparar_target_delta_reg(df_infer_base, _ridge)
    else:
        raise ValueError(target_name)
    return tr_out, inf_out, kind


def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, "La CLI de kaggle no esta instalada.  pip install kaggle"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


def correr_combo(target_name, metodo):
    tag_esc = metodo or 'fijo'
    experimento = f"{PARAM['prefijo_experimento']}__{target_name}__{tag_esc}"
    dir_out = RUTA_EXP / experimento

    if (dir_out / 'resultado.json').exists():
        print(f"[{experimento}] ya tiene resultado.json -> RESUME (se saltea)")
        return _leer_json_reintentando(dir_out / 'resultado.json')

    dir_out.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*70}\n{experimento}\n{'='*70}")

    tr_pl, inf_pl, kind = preparar_combo(target_name, metodo, df_sup_universo, df_infer_universo)
    df_pd = _a_pandas(tr_pl, _cols_pd + ['y'])
    df_infer_pd = _a_pandas(inf_pl, _cols_pd)

    db_local = Path.home() / f"optuna_{experimento}.db"
    db_bucket = RUTA_EXP / "optuna_db" / f"{experimento}.db"
    db_bucket.parent.mkdir(parents=True, exist_ok=True)
    if db_bucket.exists() and not db_local.exists():
        shutil.copy(db_bucket, db_local)
    storage = f"sqlite:///{db_local}"

    def objective(trial):
        params = espacio_hiper(trial, PARAM['techo_arboles'], PARAM['regularizacion'], PARAM['semilla'])
        modelo = entrenar(df_pd, params, MESES_TRAIN, eval_meses=MESES_VAL,
                          early_stopping_rounds=PARAM['early_stopping_rounds'])
        score, _ = evaluar_combo(modelo, df_pd, MESES_VAL, kind)
        if np.isnan(score):
            raise optuna.TrialPruned()
        bi = getattr(modelo, 'best_iteration_', None)
        trial.set_user_attr('best_iteration', int(bi) if bi else int(params['n_estimators']))
        return float(score)

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
                                study_name=experimento, storage=storage, load_if_exists=True)
    print(f"  trials previos: {len(study.trials)}  ->  corriendo {PARAM['n_trials']} nuevos")

    def respaldar():
        try:
            tmp = db_bucket.with_suffix('.db.tmp')
            shutil.copy(db_local, tmp)
            tmp.replace(db_bucket)
        except Exception as e:
            print(f"    [aviso] no se pudo respaldar: {e}")

    cada = max(1, int(PARAM['backup_cada_n_trials']))
    with tqdm(total=PARAM['n_trials'], desc=experimento[:40]) as pbar:
        def cb(st, tr):
            pbar.update(1)
            try:
                pbar.set_postfix({'mejor WAPE': f"{st.best_value:.5f}"})
            except ValueError:
                pass
            if pbar.n % cada == 0:
                respaldar()
        study.optimize(objective, n_trials=PARAM['n_trials'], callbacks=[cb])
    respaldar()
    print(f"  mejor WAPE (val): {study.best_value:.5f}")

    mejores = espacio_hiper(optuna.trial.FixedTrial(study.best_params),
                            PARAM['techo_arboles'], PARAM['regularizacion'], PARAM['semilla'])
    n_arboles_final = int(study.best_trial.user_attrs.get('best_iteration') or PARAM['techo_arboles'])
    mejores_fijos = dict(mejores); mejores_fijos['n_estimators'] = n_arboles_final

    modelo_val = entrenar(df_pd, mejores_fijos, MESES_TRAIN)
    wape_val, _ = evaluar_combo(modelo_val, df_pd, MESES_VAL, kind)

    meses_fit_test = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
    modelo_test = entrenar(df_pd, mejores_fijos, meses_fit_test)
    wape_test, _ = evaluar_combo(modelo_test, df_pd, MESES_TEST, kind)
    print(f"  VAL wape={wape_val:.5f}   TEST wape={wape_test:.5f}")

    def _wape_naive(meses):
        _n = df_pd[df_pd['periodo'].isin(meses)]
        return wape(_n['clase_tn'], _n['tn0'] if 'tn0' in _n.columns else _n['clase_tn'] * 0, _n['product_id']) \
              if len(_n) else float('nan')
    naive_val, naive_test = _wape_naive(MESES_VAL), _wape_naive(MESES_TEST)

    meses_fit_final = sorted(set(df_pd['periodo'].unique().tolist()))
    modelo_final = entrenar(df_pd, mejores_fijos, meses_fit_final)

    pred_infer_y = modelo_final.predict(df_infer_pd[FEATURES])
    tn_pred_universo = reconstruir_target(pred_infer_y, df_infer_pd, kind)
    pred_infer = df_infer_pd[['product_id', 'customer_id', 'periodo', 'periodo_objetivo']].copy()
    pred_infer['tn_pred'] = np.maximum(tn_pred_universo, PARAM['clip_min'])

    OBJ = PARAM['periodo_objetivo']
    combinado = pl.concat([
        pl.from_pandas(pred_infer).select('product_id', 'periodo_objetivo',
                                          pl.col('tn_pred').alias('tn')),
        FALLBACK_INFER.select('product_id', 'periodo_objetivo', pl.col('tn_pred_fallback').alias('tn')),
    ], how='vertical_relaxed')
    obj = combinado.filter(pl.col('periodo_objetivo') == OBJ)
    if obj.is_empty():
        raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                           f"{sorted(combinado['periodo_objetivo'].unique().to_list())}")
    por_producto = obj.group_by('product_id').agg(pl.col('tn').sum().alias('tn'))
    submit = (oficiales.select('product_id').join(por_producto, on='product_id', how='left'))
    sin_pred = int(submit['tn'].null_count())
    submit = submit.with_columns(pl.col('tn').fill_null(0.0)).sort('product_id')

    path_submit = dir_out / f"submission_{OBJ}.csv"
    submit.write_csv(path_submit)
    print(f"  submit: {submit.height} filas, {sin_pred} sin prediccion, tn total {submit['tn'].sum():,.1f}")

    resultado = {
        'experimento': experimento, 'target': target_name, 'escalado': metodo, 'kind': kind,
        'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
        'periodo_objetivo': OBJ,
        'wape_val': wape_val, 'wape_test': wape_test,
        'wape_naive_val': naive_val, 'wape_naive_test': naive_test,
        'n_trials_total': len(study.trials), 'n_estimators_final': n_arboles_final,
        'hiperparametros': study.best_params,
        'n_filas_universo': int(tr_pl.height), 'n_pares_universo': int(UNIVERSO.height),
        'n_features': len(FEATURES), 'n_features_hoja': len(FEATURES_SOBREVIVEN_HOJA),
        'sin_prediccion': sin_pred, 'tn_total_entregado': float(submit['tn'].sum()),
        'path_submit': str(path_submit.relative_to(BUCKET)),
        'semilla': PARAM['semilla'],
    }
    with open(dir_out / 'resultado.json', 'w', encoding='utf-8') as f:
        json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)
    shutil.copy(db_local, db_bucket)
    print(f"  guardado en {dir_out.relative_to(BUCKET)}")
    return resultado


print("correr_combo() listo")


## 10) Correr la grilla completa (resumible, identico a `01_`)


In [ ]:
t0 = time.time()
resultados = []
for target_name, metodo in GRILLA:
    r = correr_combo(target_name, metodo)
    resultados.append(r)

manifest = {
    'grilla': [{'target': t, 'escalado': e} for t, e in GRILLA],
    'n_combos': len(GRILLA),
    'n_features': len(FEATURES),
    'resultados': [
        str((RUTA_EXP / f"{PARAM['prefijo_experimento']}__{r['target']}__{r['escalado'] or 'fijo'}"
            / 'resultado.json').relative_to(BUCKET))
        for r in resultados
    ],
}
with open(RUTA_EXP / f"manifest_{PARAM['prefijo_experimento']}.json", 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, default=str)
print(f"\nGrilla completa: {len(resultados)}/{len(GRILLA)} combos.   [{time.time()-t0:.0f}s]")


## 11) Submit — TODOS los combos que terminaron, a Kaggle, pase lo que pase


In [ ]:
if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube nada. Los CSV ya estan generados.")
else:
    kdst = Path.home() / ".kaggle" / "kaggle.json"
    kdst.parent.mkdir(parents=True, exist_ok=True)
    if not kdst.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kdst); kdst.chmod(0o600)
                print(f"Kaggle auth copiada de {cand}")
                break
    if not kdst.exists():
        print("Sin credenciales de Kaggle: no se sube nada. Los CSV ya estan generados.")
    else:
        kdst.chmod(0o600)
        print(f"Subiendo {len(resultados)} submits a Kaggle (sin filtrar por cuota)...")
        for i, r in enumerate(resultados, 1):
            path_submit = BUCKET / r['path_submit']
            msg = f"{r['experimento']} | wape_test={r['wape_test']:.5f}"
            ok, salida = kaggle_cli(["competitions", "submit", "-c", PARAM['kaggle_competition'],
                                    "-f", str(path_submit), "-m", msg])
            print(f"  [{i}/{len(resultados)}] {r['experimento']}: " + ("OK" if ok else "FALLO"))
            if not ok:
                print(f"    {salida[-300:]}")
            time.sleep(PARAM['pausa_entre_submits_seg'])


## 12) Leaderboard — combinado con `01_` (columna `version`)

Mismo archivo `leaderboard_otro_pipe.csv` que ya escribe `01_` -- se le
agrega una columna `version` (`v1`/`v2`) para poder comparar ambas rondas
juntas de un vistazo.


In [ ]:
filas = []
for r in resultados:
    _nt = r.get('wape_naive_test')
    filas.append({
        'experimento': r['experimento'], 'version': 'v2', 'target': r['target'], 'escalado': r['escalado'],
        'wape_val': round(r['wape_val'], 5), 'wape_test': round(r['wape_test'], 5),
        'wape_naive_test': round(_nt, 5) if _nt == _nt else None,
        'brecha_test_val': round(r['wape_test'] - r['wape_val'], 5),
        'sin_prediccion': r['sin_prediccion'], 'tn_entregado': round(r['tn_total_entregado'], 1),
    })
nueva = pl.DataFrame(filas)

path_lb = RUTA_EXP / 'leaderboard_otro_pipe.csv'
if path_lb.exists():
    viejo = pl.read_csv(path_lb)
    if 'version' not in viejo.columns:
        viejo = viejo.with_columns(pl.lit('v1').alias('version'))
    viejo = viejo.filter(~pl.col('experimento').is_in(nueva['experimento'].to_list()))
    nueva = pl.concat([viejo, nueva], how='diagonal_relaxed')
nueva = nueva.sort('wape_test')
nueva.write_csv(path_lb)
print(f"leaderboard_otro_pipe.csv ({nueva.height} combos, V1+V2 combinados):")
print(nueva)


## 13) Mezcla de los combos que mejor generalizan (NUEVO)

Promedio simple de las predicciones de los combos con menor brecha
val->test (mas confiables, no solo el mejor WAPE test individual) --
combinar modelos poco correlacionados entre si suele reducir varianza y
igualar o superar al mejor modelo individual. Editar
`PARAM['combos_para_mezcla']` segun lo que diga el leaderboard real (los 4
de aca abajo son un default, no una regla fija) -- tiene que haber
`resultado.json` para cada uno (o sea, que ya hayan corrido).


In [ ]:
PARAM['combos_para_mezcla'] = [
    f"{PARAM['prefijo_experimento']}__log_ton_norm__normalpower",
    f"{PARAM['prefijo_experimento']}__log_ton_norm__media",
    f"{PARAM['prefijo_experimento']}__delta_mean_12__fijo",
    f"{PARAM['prefijo_experimento']}__delta_reg__fijo",
]

_faltantes = [e for e in PARAM['combos_para_mezcla']
             if not (RUTA_EXP / e / 'resultado.json').exists()]
if _faltantes:
    raise FileNotFoundError(f"Faltan corrida(s): {_faltantes}. Corre esos combos primero "
                            f"(o ajusta PARAM['combos_para_mezcla'] a los que si tenes).")

_dfs_mezcla = []
for e in PARAM['combos_para_mezcla']:
    _res = _leer_json_reintentando(RUTA_EXP / e / 'resultado.json')
    _csv = pl.read_csv(BUCKET / _res['path_submit'])
    print(f"  {e:45s} wape_val={_res['wape_val']:.5f}  wape_test={_res['wape_test']:.5f}  "
         f"tn_total={_csv['tn'].sum():,.1f}")
    _dfs_mezcla.append(_csv.rename({'tn': f'tn__{e}'}))

_mezcla = _dfs_mezcla[0]
for _df in _dfs_mezcla[1:]:
    _mezcla = _mezcla.join(_df, on='product_id', how='inner')
assert _mezcla.height == _dfs_mezcla[0].height, "el join entre combos perdio filas -- revisar"

_cols_tn = [c for c in _mezcla.columns if c.startswith('tn__')]
_mezcla = _mezcla.with_columns(pl.mean_horizontal(_cols_tn).alias('tn')).select('product_id', 'tn')

OBJ = PARAM['periodo_objetivo']
path_mezcla = RUTA_EXP / f"submission_{OBJ}_mezcla_top{len(PARAM['combos_para_mezcla'])}.csv"
_mezcla.write_csv(path_mezcla)
print(f"\nmezcla de {len(PARAM['combos_para_mezcla'])} combos: {_mezcla.height} filas, "
     f"tn total {_mezcla['tn'].sum():,.1f}")
print(f"Guardado: {path_mezcla}")


### Submit de la mezcla a Kaggle (opcional)


In [ ]:
if not PARAM['submit']:
    print("PARAM['submit'] = False -> no se sube. El CSV de la mezcla ya esta generado.")
else:
    kdst = Path.home() / ".kaggle" / "kaggle.json"
    kdst.parent.mkdir(parents=True, exist_ok=True)
    if not kdst.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kdst); kdst.chmod(0o600)
                break
    if not kdst.exists():
        print("Sin credenciales de Kaggle: no se sube. El CSV ya esta generado.")
    else:
        kdst.chmod(0o600)
        msg = f"mezcla de {len(PARAM['combos_para_mezcla'])}: " + ", ".join(PARAM['combos_para_mezcla'])
        ok, salida = kaggle_cli(["competitions", "submit", "-c", PARAM['kaggle_competition'],
                                "-f", str(path_mezcla), "-m", msg[:200]])
        print(f"mensaje: {msg}\n{salida}")
        print("Submit enviado." if ok else "NO se pudo subir; el CSV esta en disco.")
